## nb_03_player_info_silver

Cleans and validates table in bronze schema and and writes the result to sliver schema
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [8]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, count, when, lit, min, max,
    substring, concat, to_timestamp, to_date,
)

StatementMeta(, 122d943a-9588-4444-b4fa-9342e7d0bcbf, 26, Finished, Available, Finished, False)

### Load common db functions

In [9]:
%run nb_00_dbutils

StatementMeta(, 122d943a-9588-4444-b4fa-9342e7d0bcbf, 35, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded:

```
%run nb_02_game_silver { "RUN_PIPELINE": false }
```

In [10]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 122d943a-9588-4444-b4fa-9342e7d0bcbf, 36, Finished, Available, Finished, False)

### Config

In [11]:
BRONZE_TABLE = "bronze.team_info"
SILVER_TABLE = "silver.team_info"

# Only these columns make it into the silver table
cols: list[str] = [
    "team_id",
    "shortName",
    "teamName",
    "abbreviation",
]

# Natural key used to de-duplicate rows
PRIMARY_KEYS: list[str] = ["team_id"]
DEDUPE_KEYS: list[str] = ["team_id"]


StatementMeta(, 122d943a-9588-4444-b4fa-9342e7d0bcbf, 37, Finished, Available, Finished, False)

## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

### Step 1 — Load bronze.game

In [12]:
if RUN_PIPELINE:
    df = load_table(spark, BRONZE_TABLE, columns=cols)

    df = remove_duplicates(df, columns=DEDUPE_KEYS)
    df = validate_no_nulls(df, columns=cols)

    # Validate primary key
    df = validate_primary_key(df, keys=PRIMARY_KEYS)

    write_table(df, SILVER_TABLE)
    print("🏁 Silver load complete.")

StatementMeta(, 122d943a-9588-4444-b4fa-9342e7d0bcbf, 38, Finished, Available, Finished, False)

✅ Loaded bronze.team_info: 33 rows
🔁 Removed 0 duplicate row(s) based on ['team_id']
✅ Primary key check passed — ['team_id'] is unique across 33 row(s)
✅ Wrote silver.team_info (33 rows, 4 columns)
🏁 Silver load complete.
